# 🏥 Insurance Risk Model — Exploration & SHAP Visualizations

This notebook covers:
1. Dataset exploration (class balance, feature distributions)
2. Model performance (AUC, confusion matrix, ROC curve)
3. SHAP explanations (summary plot, waterfall, force plot)
4. Single-user prediction walkthrough

In [ ]:
import os, sys, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import shap
import joblib
from sklearn.metrics import (
    roc_auc_score, roc_curve,
    confusion_matrix, ConfusionMatrixDisplay
)

warnings.filterwarnings('ignore')
shap.initjs()

# Add project root to path
sys.path.insert(0, os.path.join(os.getcwd(), '..'))
from predict_risk import predict_risk, DIABETES_FEATURES, CARDIAC_FEATURES, _build_diabetes_row, _build_cardiac_row, _scale_input
from train_models import load_diabetes_data, load_cardiac_data

MODELS_DIR = os.path.join('..', 'models')
print('✅ Imports done')

## 1. Load Data & Models

In [ ]:
X_d, y_d = load_diabetes_data()
X_c, y_c = load_cardiac_data()

d_model = joblib.load(os.path.join(MODELS_DIR, 'diabetes_model.pkl'))
c_model = joblib.load(os.path.join(MODELS_DIR, 'cardiac_model.pkl'))

print(f'Diabetes: {X_d.shape} | Cardiac: {X_c.shape}')

## 2. Class Balance

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(10, 4))
for ax, y, title in zip(axes, [y_d, y_c], ['Diabetes', 'Cardiac']):
    counts = y.value_counts()
    ax.bar(['No Disease', 'Disease'], counts.values, color=['#4C9BE8','#E84C4C'])
    ax.set_title(f'{title} — Class Balance')
    ax.set_ylabel('Count')
    for i, v in enumerate(counts.values):
        ax.text(i, v + 5, str(v), ha='center', fontweight='bold')
plt.tight_layout()
plt.savefig('class_balance.png', dpi=150)
plt.show()

## 3. Feature Distributions

In [ ]:
# Diabetes feature distributions split by outcome
fig, axes = plt.subplots(2, 4, figsize=(16, 8))
axes = axes.flatten()
df_d = X_d.copy()
df_d['outcome'] = y_d.values

for i, feat in enumerate(DIABETES_FEATURES):
    ax = axes[i]
    for label, color in [(0, '#4C9BE8'), (1, '#E84C4C')]:
        subset = df_d[df_d['outcome'] == label][feat].dropna()
        ax.hist(subset, bins=25, alpha=0.6, color=color,
                label='No Diabetes' if label == 0 else 'Diabetes')
    ax.set_title(feat)
    ax.legend(fontsize=7)
plt.suptitle('Diabetes — Feature Distributions by Outcome', fontsize=13, y=1.01)
plt.tight_layout()
plt.savefig('diabetes_distributions.png', dpi=150)
plt.show()

## 4. Model Performance — ROC Curves

In [ ]:
from sklearn.model_selection import train_test_split

fig, axes = plt.subplots(1, 2, figsize=(12, 5))

for ax, model, X, y, title in zip(
    axes,
    [d_model, c_model],
    [X_d, X_c],
    [y_d, y_c],
    ['Diabetes Model', 'Cardiac Model']
):
    _, X_test, _, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)
    y_prob = model.predict_proba(X_test)[:, 1]
    fpr, tpr, _ = roc_curve(y_test, y_prob)
    auc = roc_auc_score(y_test, y_prob)
    ax.plot(fpr, tpr, color='#E84C4C', lw=2, label=f'AUC = {auc:.3f}')
    ax.plot([0,1],[0,1],'k--', lw=1)
    ax.set_xlabel('False Positive Rate')
    ax.set_ylabel('True Positive Rate')
    ax.set_title(f'{title} — ROC Curve')
    ax.legend()

plt.tight_layout()
plt.savefig('roc_curves.png', dpi=150)
plt.show()

## 5. Confusion Matrices

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(10, 4))

for ax, model, X, y, title in zip(
    axes,
    [d_model, c_model],
    [X_d, X_c],
    [y_d, y_c],
    ['Diabetes', 'Cardiac']
):
    _, X_test, _, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)
    y_pred = model.predict(X_test)
    cm = confusion_matrix(y_test, y_pred)
    disp = ConfusionMatrixDisplay(cm, display_labels=['No Disease', 'Disease'])
    disp.plot(ax=ax, colorbar=False, cmap='Blues')
    ax.set_title(f'{title} — Confusion Matrix')

plt.tight_layout()
plt.savefig('confusion_matrices.png', dpi=150)
plt.show()

## 6. SHAP Summary Plots

In [ ]:
# Helper to extract XGB from pipeline
def get_xgb(pipeline):
    for _, step in pipeline.steps:
        if hasattr(step, 'get_booster'):
            return step
    raise ValueError('No XGB in pipeline')

def get_scaler_output(pipeline, X):
    for _, step in pipeline.steps:
        if hasattr(step, 'transform') and not hasattr(step, 'fit_resample'):
            return step.transform(X)
    return X.values

# ── Diabetes SHAP summary ─────────────────────────────────────────────────────
X_d_scaled = get_scaler_output(d_model, X_d)
d_explainer = shap.TreeExplainer(get_xgb(d_model))
d_shap_vals = d_explainer.shap_values(X_d_scaled)

plt.figure(figsize=(8, 5))
shap.summary_plot(d_shap_vals, X_d, feature_names=DIABETES_FEATURES, show=False)
plt.title('Diabetes Model — SHAP Feature Importance')
plt.tight_layout()
plt.savefig('shap_diabetes_summary.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved shap_diabetes_summary.png')

In [ ]:
# ── Cardiac SHAP summary ──────────────────────────────────────────────────────
X_c_scaled = get_scaler_output(c_model, X_c)
c_explainer = shap.TreeExplainer(get_xgb(c_model))
c_shap_vals = c_explainer.shap_values(X_c_scaled)

plt.figure(figsize=(8, 5))
shap.summary_plot(c_shap_vals, X_c, feature_names=CARDIAC_FEATURES, show=False)
plt.title('Cardiac Model — SHAP Feature Importance')
plt.tight_layout()
plt.savefig('shap_cardiac_summary.png', dpi=150, bbox_inches='tight')
plt.show()

## 7. Single-User Prediction & SHAP Waterfall

In [ ]:
sample_user = {
    'age': 35, 'sex': 'F', 'bmi': 32.5, 'blood_pressure': 88,
    'glucose': 165, 'insulin': 180, 'skin_thickness': 35,
    'diabetes_pedigree_function': 0.627, 'pregnancies': 1,
    'cholesterol': 240, 'chest_pain_type': 2,
    'fasting_blood_sugar_high': 0, 'resting_ecg': 0,
    'max_heart_rate': 150, 'exercise_angina': 0, 'st_depression': 1.0,
}

result = predict_risk(sample_user)

print('=== DIABETES ===')
print(f"  Score : {result['diabetes']['risk_score']}  ({result['diabetes']['risk_level']})")
print(f"  Reason: {result['diabetes']['explanation']}")
print(f"  SHAP  : {result['diabetes']['feature_contributions']}")

print('\n=== CARDIAC ===')
print(f"  Score : {result['cardiac']['risk_score']}  ({result['cardiac']['risk_level']})")
print(f"  Reason: {result['cardiac']['explanation']}")
print(f"  SHAP  : {result['cardiac']['feature_contributions']}")

In [ ]:
# SHAP waterfall for the sample user — Diabetes
d_row    = _build_diabetes_row(sample_user, d_model.feature_medians_)
d_scaled = _scale_input(d_model, d_row)
d_sv     = d_explainer.shap_values(d_scaled)

exp = shap.Explanation(
    values=d_sv[0],
    base_values=d_explainer.expected_value,
    data=d_row.values[0],
    feature_names=DIABETES_FEATURES
)
shap.waterfall_plot(exp, show=True)

In [ ]:
# Bar plot of feature contributions — easy to embed in UI
import matplotlib.patches as mpatches

def plot_contributions(contributions: dict, title: str, ax):
    feats  = list(contributions.keys())
    vals   = list(contributions.values())
    colors = ['#E84C4C' if v > 0 else '#4C9BE8' for v in vals]
    bars   = ax.barh(feats, vals, color=colors)
    ax.axvline(0, color='black', lw=0.8)
    ax.set_title(title)
    ax.set_xlabel('SHAP value (impact on risk score)')
    red_p  = mpatches.Patch(color='#E84C4C', label='Increases risk')
    blue_p = mpatches.Patch(color='#4C9BE8', label='Decreases risk')
    ax.legend(handles=[red_p, blue_p], fontsize=8)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
plot_contributions(result['diabetes']['feature_contributions'], 'Diabetes Risk — Feature Contributions', axes[0])
plot_contributions(result['cardiac']['feature_contributions'],  'Cardiac Risk — Feature Contributions',  axes[1])
plt.tight_layout()
plt.savefig('feature_contributions.png', dpi=150)
plt.show()
print('Saved feature_contributions.png')

## 8. Insurance Plan Matching

In [ ]:
sys.path.insert(0, '..')
from insurance_matcher import match_plans

plans = match_plans(result, user_profile={'age': 35}, top_n=5)

print('\n🏥 Top Recommended Plans:\n')
for i, p in enumerate(plans, 1):
    print(f"{i}. [{p['suitability_score']:.2f}] {p['name']} — {p['insurer']} ({p['type']})")
    print(f"   Cover: ₹{p['coverage_limit']:,}  |  Premium: ₹{p['premium_range'][0]:,}–₹{p['premium_range'][1]:,}/yr")
    print(f"   ✅ {p['recommendation_reason']}\n")